# FORESEE - DM-Electron Scattering with a Dark Photon Mediator

### Load Libraries 

In [ ]:
import sys, os
src_path = "../../"
sys.path.append(src_path)
import numpy as np
from src.foresee import Foresee, Utility, Model
from src.utils.utility import BREM_MASSES
from matplotlib import pyplot as plt

In [ ]:
#Create symlink to direct production spectra.
import os
try: os.unlink('model/direct')
except: pass
os.symlink(
    src=os.path.normpath('../../../files/direct/DM-electron'),
    dst='model/direct',
    target_is_directory=True,
)

## 1. Specifying the Model

The phenomenology of complex scalar dark matter $\chi$ that is mediated by a dark photon $A'$ can be described by the following Lagrangian

\begin{equation}
 \mathcal{L} = |\partial_\mu \chi|^2 - m^2_\chi |\chi|^2 + \frac{1}{2} \textcolor{red}{m_{A'}} A'^2 + A'_\mu ( e \textcolor{red}{\epsilon} J_{EM}^\mu + \textcolor{red}{g_D} J_D^\mu)
\end{equation}

with the DM mass $m_\chi$, the dark photon mass $m_{A'}$, the kinetic mixing $\epsilon$ and the dark coupling $g_D$ as free parameters. In the following we will fix $m_{A'}=3m_\chi$ and $\alpha_D = g_D^2/4\pi = 0.5$. For the search for dark matter scattering at forward experiments we need to know i) the *production rate*, and ii) the *interaction rate*. All these properties are specified in the `Model` class. We initialize it with the name of the model as argument. 

In [ ]:
energy = "14"
modelname="DM-electron"
model = Model(modelname)

# Builder parameters, matching the build.py / load_model() defaults.
nsample_3body = 2000
generators_light = ["EPOSLHC", "SIBYLL", "QGSJET"][:1]
brem_configurations = ["Brem_QRA_L1.5", "Brem_QRA_L1.0", "Brem_QRA_L2.0"][:1]

**Production** In the simplest case, the DM is produced through the decay of on-shell dark photons in the process $M\to\gamma A'\to \gamma\chi\bar{\chi}$ with $M = \pi, \eta$. The corresponding branching fractions for these decay chains are given by
\begin{align}
    \text{BR}(M \to A' \gamma) &= 2 \epsilon^2 \times\text{BR}(M \to \gamma\gamma) \times \left(1-m_{A'}^2/m_M^2\right)^3 \\ \\
    \text{BR}(A'\to \chi\bar{\chi}) &\approx 1 \quad\quad (\alpha_D \gg \alpha\epsilon^2).
\end{align}
Since we have two $\chi$'s in the final state, we multiply by an additional factor 2.

In [ ]:
model.add_production_3bodydecay(
    pid0 = "111",
    pid1 = "22",
    pid2 = "0",
    br = ["2*2.*0.99 * coupling**2 * pow(1.-pow(3*mass/self.masses('111'),2),3)","3*mass"],
    generator = generators_light,
    energy = energy,
    nsample = nsample_3body,
    integration = "chain_decay",
    massrange=[0,0.135/3.],
)

model.add_production_3bodydecay(
    pid0 = "221",
    pid1 = "22",
    pid2 = "0",
    br = ["2*2.*0.39 * coupling**2 * pow(1.-pow(3*mass/self.masses('221'),2),3)","3*mass"],
    generator = generators_light,
    energy = energy,
    nsample = nsample_3body,
    integration = "chain_decay",
    massrange=[0,0.547862/3.],
)

DM can also be produced via the decay of dark photons produced in dark Bremsstrahlung, so coherent radiation off a proton in processes such as $p p \to p p A'\to p p \chi\bar{\chi}$. The spectra for dark photons was obtained following the description in [1708.09389](https://arxiv.org/abs/1708.09389), and subsequently decayed to produce the DM spectra, which is provided in the `model/direct` directory.


In [ ]:
# Bremsstrahlung A'-mass grid (shared with DarkPhoton et al.). DM-electron
# indexes its precomputed spectra by m_chi = m_{A'}/3, so we divide by 3.
masses_brem = [round(m / 3, 6) for m in BREM_MASSES]

model.add_production_direct(
    label = "Brem",
    energy = energy,
    configuration = brem_configurations,
    coupling_ref=1,
    masses = masses_brem,
)


At high masses, the production of dark photons is dominated by Drell-Yan production, so quark-anti-quark fusion $q q \to A' X\to \chi\bar{\chi}X$. The spectra for DM at some reference coupling needs to be provided in the `model/direct` directory. 

In [ ]:
masses_dy = [0.5283, 0.592767, 0.6651, 0.746233, 0.8373, 0.939467, 1.0541, 1.327033, 
            1.670633, 2.1032, 2.647767, 3.333333, 4.0, 5.0, 5.666667, 6.666667, 8.333333, 
             10.0, 16.666667, 23.333333, 33.333333] 


model.add_production_direct(
    label = "DY",
    energy = energy,
    coupling_ref=1,
    masses = masses_dy,
)

**Interaction Rate:** DM with a dark photon mediator can scatter off of electrons resulting in electron recoil signatures. Following [2101.10338](https://arxiv.org/abs/2101.10338), the differential scattering rate with respect to the recoil energy is given by 
\begin{equation}
 \frac{d\sigma}{d E_R} 
 = \frac{8 \pi \epsilon^2 \alpha \alpha_D  m_e }{(m_{A'}^2 + 2 m_e E_R)^2}
\end{equation}

In [ ]:
model.set_dsigma_drecoil_1d(
    dsigma_der="coupling**2 * 8 * 3.1415 * 1./137. * 0.5 * 0.000511 / (mass**2 * 3**2 +2*0.000511*recoil)**2", 
    recoil_max = "2 * 0.000511 * (energy**2-mass**2) / (0.000511*(2*energy+mass) + mass**2)",
    coupling_ref=1
)

We can now initiate FORESEE with the model that we just created. 

In [ ]:
foresee = Foresee(path=src_path)
foresee.set_model(model=model)

## 3. Event Generation

In the following, we want to study one specific benchmark point with $m_{\chi}=100$ MeV and $\epsilon = 1$. 

In [ ]:
mass, coupling, = 0.1, 1.0

First, we will produce the corresponding flux for this mass and a reference coupling $\epsilon_{ref}=1$. 

In [ ]:
%%time
plot=foresee.get_llp_spectrum(mass=mass, coupling=1, do_plot=True)
os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Spectrum_{modelname}.pdf", bbox_inches="tight")
plot.show()

Next, let us define the configuration of the detector (in terms of position, size and luminosity). Here we choose FLArE at the FPF. 

In [ ]:
foresee.set_detector(
    distance= 620, 
    selection= "abs(x.x)<0.5 and abs(x.y)<0.5", 
    length= 7, 
    luminosity= 3000, 
    numberdensity= 3.754e+29, #LAr
    ermin= 0.03, 
    ermax= 1
)

For our benchmark point, let us now look at how many particles interact inside the detector volume. 

In [ ]:
setupnames  =  generators_light

_, _, momenta, weights = foresee.get_events_interaction(
    mass = mass, 
    energy=energy, 
    couplings= [coupling], 
    nsample = 1
)

for isetup, setup in enumerate(setupnames):
    print("Expected number of events for "+setup+":", round(sum(weights[0][:,isetup]),7))

Let us plot the resulting energy distribution.

In [ ]:
fig = plt.figure(figsize=(7,5))
ax = plt.subplot(1,1,1)
energies = [p.e for p in momenta], 
for isetup, setup in enumerate(setupnames):
    ax.hist(energies, weights=weights[0][:,isetup], bins=np.logspace(2,4, 20+1), histtype='step', label=setup) 
ax.set_xscale("log")
ax.set_xlim(1e2,1e4) 
ax.set_xlabel("E [GeV]") 
ax.set_ylabel("Number of Events per Bin") 
ax.legend(frameon=False, labelspacing=0, fontsize=14, loc='upper right')
os.makedirs(f"figures/{modelname}", exist_ok=True)
plt.savefig(f"figures/{modelname}/E_distribution_{modelname}.pdf", bbox_inches="tight")
plt.show()

## 3. Sensitivity Reach

In the following, we will obtain the projected sensitivity for the LLP model. For this, we first define a grid of couplings and masses, and then produce the corresponding fluxes. 

In [ ]:
masses= [round(x,5) for x in np.logspace(-3,0,50)]
thresholds = [
    0.13093, 0.13498, 0.13903, 0.53143, 0.54786, 0.5643,
]
masses = sorted(masses + thresholds)
couplings = np.logspace(-5,-2,100)

# Use cached LLP spectra: get_llp_spectrum recomputes on every call,
# so skip any masses already saved in model/LLP_spectra/.
for mass in masses:
    if not os.path.exists(f"model/LLP_spectra/{energy}TeV_m_{mass}.txt.gz"):
        foresee.get_llp_spectrum(mass=mass, coupling=1)

We can now plot the `production rate vs mass` using the `foresee.plot_production()` function.

In [ ]:
productions=[
    {"channels": ["111"] , "color": "red"      , "label": r"$\pi^0 \to \gamma A'$", "generators": generators_light  },
    {"channels": ["221"] , "color": "orange"   , "label": r"$\eta \to \gamma A'$" , "generators": generators_light  },
    {"channels": ["Brem"], "color": "limegreen", "label": r"Bremsstrahlung"       , "generators": brem_configurations},
    {"channels": ["DY"]  , "color": "tab:purple", "label": r"Drell-Yan"            , "generators": ["DY"]},    
]

plot=foresee.plot_production(
    masses = masses,
    productions = productions,
    energy=energy,
    condition="logth<-3 and logp>2", 
    xlims=[0.001,1],ylims=[4e7,4e11],
    xlabel=r"Mass [GeV]",
    ylabel=r"Production Rate $\sigma/\epsilon^2$ [pb]",
    title=r"$\theta < 1$ mrad and $E > 100$ GeV",
    legendloc=(0.97,1),
    fs_label=12,
    ncol=2,
    figsize=(7,6),
)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Production_{modelname}.pdf", bbox_inches="tight")
plot.show()

Let us now scan over various masses and couplings, and record the resulting number of events. Note that here we again consider the FLArE configuration, which we set up before.

In [ ]:
configuration = 'FLArE'

condition = f"np.sqrt(p**2 + mass**2) > 100"
if not all(os.path.exists(f"model/results/{energy}TeV_{configuration}_{label}.npy") for label in setupnames):
    
    list_nevents = {label:[] for label in setupnames}
    for mass in masses:
        _, nevents, _, _ = foresee.get_events_interaction(mass = mass, energy=energy, couplings= couplings, nsample = 5, preselectioncuts = condition)
        for i,label in enumerate(setupnames): list_nevents[label].append(nevents.T[i])  
    
    #save results
    os.makedirs(f"model/results", exist_ok=True)
    for label in setupnames: 
        result = np.array([masses,couplings,list_nevents[label]], dtype='object')
        np.save("model/results/"+energy+"TeV_"+configuration+"_"+label+".npy",result)

Now let's plot the results. We first specify all detector setups for which we want to show result (filename in model/results directory, label, color, linestyle, opacity alpha for filled contours, required number of events).

In [ ]:
setups = [
    ["14TeV_FLArE_EPOSLHC.npy"      , "FLArE"  , "red"      ,     "solid" , 0., 3],
]

Then we specify all the existing bounds (filename in model/bounds directory, label, label position x, label position y, label rotation)

In [ ]:
bounds = [ 
    ["bounds_BeamDumps.txt", "Fixed Target"     , 0.0012, 7.0e-4, 0],
    ["bounds_BaBar.txt"    , "BaBar"            , 0.0012, 1.5e-3, 0],
]

We then specify other projected sensitivitities (filename in model/bounds directory, color, label, label position x, label position y, label rotation)

In [ ]:
projections = [
    # ["limits_LDMX.txt",       "deepskyblue",  "LDMX"      , 0.090, 2.3e-5, 0  ],
    # ["limits_NA64.txt",       "dodgerblue",   "NA64"      , 0.010, 1.8e-5, 0  ],
    # ["limits_Belle2.txt",     "blue",         "Belle2"    , 0.250, 3.0e-5, 0  ],
    # ["limits_FLARE-elec.txt", "magenta",      ""          , 0.250, 3.0e-5, 0  ],
]

Finally, we can plot everything using `foresee.plot_reach()`. It returns a matplotlib instance, to which we can add further lines and which we can show or save. Below, we add the dark matter relict target line for a specific benchmark.

In [ ]:
plot = foresee.plot_reach(
    setups=setups,
    bounds=bounds,
    projections=[],
    lines=[
        # Scalar-DM relic-density target (arXiv:2105.07077).
        ["target_DM_scalar.txt", "k", 2, [
            ["Scalar",            1.7e-3, 9.9*1e-6, 42]
        ]],
        ["target_DM_majorana.txt", "k", 2, [
            ["Majorana",          2.7e-3, 8.0*1e-6, 42]
        ]]
    ],
    title="DM-e Scattering", 
    xlims=[0.001,1], 
    ylims=[3e-6,2e-3],
    xlabel=r"Particle Mass $m_{\chi}$ [GeV]", 
    ylabel=r"Coupling $\epsilon$",
    legendloc=(0.95,0.25),
    linewidths=2,
)

# #projections
# for file, color, label, posx, posy, rotation in projections:
#     data = foresee.readfile("model/lines/"+file)
#     plot.plot(data.T[0],data.T[1], color=color, lw=1, ls="dashed")
#     plot.text(posx, posy, label,fontsize=15,color=color,rotation=rotation)

# #DM Relic (Scalar)
# data = foresee.readfile("model/lines/target_DM_scalar.txt")
# plot.plot(data.T[0],data.T[1], color="k", lw=1, ls="solid")
# plot.text(1.7e-3, 9.9*1e-6, "Scalar",fontsize=15,color="k",rotation=42)

# #DM Relic (Majorana)
# data = foresee.readfile("model/lines/target_DM_majorana.txt")
# plot.plot(data.T[0],data.T[1], color="k", lw=1, ls="dashed")
# plot.text(2.7e-3, 8.0*1e-6, "Majorana",fontsize=15,color="k",rotation=42)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Reach_{modelname}.pdf", bbox_inches="tight")
plot.show()